In [36]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os
import glob
import json
import time
import shutil
from collections import Counter

import numpy as np

print("Imports OK")


Imports OK


In [40]:
# ============================================================
# 2. KAGGLE CONFIGURATION
# ============================================================
# ------------------------------------------------------------------
# EDIT THESE TWO LINES ONLY.
#
# They must point at the *mounted* dataset folders that appear under
# /kaggle/input/ once both datasets are attached to this notebook
# (Notebook -> Add Input -> search for your two SemanticKITTI datasets).
#
# The two datasets are uploaded by different authors, so their slugs
# are NOT assumed here -- fill in the real paths after attaching them.
# If unsure of the exact folder name, run the verification cell below;
# it will list everything currently mounted under /kaggle/input/.
# ------------------------------------------------------------------
POINTCLOUD_ROOT = "/kaggle/input/datasets/hbenallal/semantickitti/dataset"   # <-- EDIT ME
LABEL_ROOT      = "/kaggle/input/datasets/sairaudhrankomuroju/sematic-kitti/dataset"         # <-- EDIT ME

# SemanticKITTI sequence to process (two-digit string, e.g. "00" .. "21")
SEQUENCE = "00"

# ------------------------------------------------------------------
# Derived, read-only input paths -- do not edit directly.
# ------------------------------------------------------------------
VELODYNE_DIR = os.path.join(POINTCLOUD_ROOT, "sequences", SEQUENCE, "velodyne")
LABEL_DIR    = os.path.join(LABEL_ROOT,      "sequences", SEQUENCE, "labels")

# ------------------------------------------------------------------
# Every generated file goes under /kaggle/working/, the only writable
# location. Nothing is ever written back into /kaggle/input/.
# ------------------------------------------------------------------
WORKING_ROOT = "/kaggle/working"
VOXEL_DIR    = os.path.join(WORKING_ROOT, "voxel_store")        # adaptive voxel features (.npz)
MAP_DIR      = os.path.join(WORKING_ROOT, "maps_2d")            # 2.5D projected maps (.npz)
SPARSE_DIR   = os.path.join(WORKING_ROOT, "propagated_labels")  # label-propagated sparse output (.npy)
RESULTS_DIR  = os.path.join(WORKING_ROOT, "results")            # run summary / metadata (.json)

for d in [VOXEL_DIR, MAP_DIR, SPARSE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("POINTCLOUD_ROOT :", POINTCLOUD_ROOT)
print("LABEL_ROOT      :", LABEL_ROOT)
print("SEQUENCE        :", SEQUENCE)
print("VELODYNE_DIR    :", VELODYNE_DIR)
print("LABEL_DIR       :", LABEL_DIR)
print()
print("Output directories (under /kaggle/working/):")
for d in [VOXEL_DIR, MAP_DIR, SPARSE_DIR, RESULTS_DIR]:
    print("  -", d)


POINTCLOUD_ROOT : /kaggle/input/datasets/hbenallal/semantickitti/dataset
LABEL_ROOT      : /kaggle/input/datasets/sairaudhrankomuroju/sematic-kitti/dataset
SEQUENCE        : 00
VELODYNE_DIR    : /kaggle/input/datasets/hbenallal/semantickitti/dataset/sequences/00/velodyne
LABEL_DIR       : /kaggle/input/datasets/sairaudhrankomuroju/sematic-kitti/dataset/sequences/00/labels

Output directories (under /kaggle/working/):
  - /kaggle/working/voxel_store
  - /kaggle/working/maps_2d
  - /kaggle/working/propagated_labels
  - /kaggle/working/results


In [41]:
# ============================================================
# 3. DATASET STRUCTURE VERIFICATION
# ============================================================
def _fail(msg):
    raise FileNotFoundError(msg)


def _sequences_with(root, leaf_name):
    """List sequence IDs under root/sequences/*/<leaf_name> that actually exist
    and are non-empty. Used to give an actionable message instead of a bare
    'not found' when SEQUENCE points at the wrong folder."""
    hits = []
    for seq_dir in sorted(glob.glob(os.path.join(root, "sequences", "*"))):
        leaf = os.path.join(seq_dir, leaf_name)
        if os.path.isdir(leaf) and os.listdir(leaf):
            hits.append(os.path.basename(seq_dir))
    return hits


print("=" * 70)
print("VERIFYING KAGGLE INPUT DATASETS")
print("=" * 70)

if not os.path.isdir(POINTCLOUD_ROOT):
    print(f"\u2717 POINTCLOUD_ROOT does not exist: {POINTCLOUD_ROOT}")
    print("  Datasets currently mounted under /kaggle/input/:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print("   -", p)
    _fail("Fix POINTCLOUD_ROOT in the configuration cell above.")
print(f"\u2713 POINTCLOUD_ROOT found : {POINTCLOUD_ROOT}")

if not os.path.isdir(LABEL_ROOT):
    print(f"\u2717 LABEL_ROOT does not exist: {LABEL_ROOT}")
    print("  Datasets currently mounted under /kaggle/input/:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print("   -", p)
    _fail("Fix LABEL_ROOT in the configuration cell above.")
print(f"\u2713 LABEL_ROOT found      : {LABEL_ROOT}")

if not os.path.isdir(VELODYNE_DIR):
    available = _sequences_with(POINTCLOUD_ROOT, "velodyne")
    _fail(f"velodyne folder not found: {VELODYNE_DIR}\n"
          f"SEQUENCE='{SEQUENCE}' has no velodyne/ folder in the point-cloud dataset.\n"
          f"Sequences with point clouds in this dataset: {available}")
print(f"\u2713 VELODYNE_DIR found    : {VELODYNE_DIR}")

# ------------------------------------------------------------------
# Labels are OPTIONAL at this stage, not required.
#
# SemanticKITTI only ships ground-truth .label files for sequences 00-10.
# Sequences 11-21 are the benchmark's test split: they have point clouds
# but NO released labels. That is expected, not an error -- so instead of
# failing here, we just detect it and record HAS_LABELS, which later
# cells use to run voxelization + 2.5D projection as normal while
# skipping the label-dependent stages (remap / propagate / sparse save).
# ------------------------------------------------------------------
HAS_LABELS = (
    os.path.isdir(LABEL_DIR)
    and len(glob.glob(os.path.join(LABEL_DIR, "*.label"))) > 0
)

if HAS_LABELS:
    print(f"\u2713 LABEL_DIR found       : {LABEL_DIR}")
else:
    available = _sequences_with(LABEL_ROOT, "labels")
    print(f"\u26a0 No labels for SEQUENCE='{SEQUENCE}' in the label dataset "
          f"({LABEL_DIR})")
    print("  This is expected for SemanticKITTI sequences 11-21 (test split,")
    print("  no released ground truth). Voxelization and 2.5D projection will")
    print("  still run on every frame; label-dependent stages (remap, voxel/")
    print("  cell label propagation, sparse label save) will be skipped.")
    print(f"  Sequences with labels in this dataset: {available}")

n_bin   = len(glob.glob(os.path.join(VELODYNE_DIR, "*.bin")))
n_label = len(glob.glob(os.path.join(LABEL_DIR, "*.label"))) if os.path.isdir(LABEL_DIR) else 0
print(f"\n  {n_bin:,} .bin files found in VELODYNE_DIR")
print(f"  {n_label:,} .label files found in LABEL_DIR")
print(f"  HAS_LABELS = {HAS_LABELS}")


VERIFYING KAGGLE INPUT DATASETS
✓ POINTCLOUD_ROOT found : /kaggle/input/datasets/hbenallal/semantickitti/dataset
✓ LABEL_ROOT found      : /kaggle/input/datasets/sairaudhrankomuroju/sematic-kitti/dataset
✓ VELODYNE_DIR found    : /kaggle/input/datasets/hbenallal/semantickitti/dataset/sequences/00/velodyne
✓ LABEL_DIR found       : /kaggle/input/datasets/sairaudhrankomuroju/sematic-kitti/dataset/sequences/00/labels

  4,541 .bin files found in VELODYNE_DIR
  4,541 .label files found in LABEL_DIR
  HAS_LABELS = True


In [42]:
# ============================================================
# 4. FRAME DISCOVERY / TEST CONFIGURATION
# ============================================================
# ------------------------------------------------------------------
# TEST_MODE = True  -> only process the first MAX_FRAMES matched frames
# TEST_MODE = False -> process every matched frame in the sequence
#
# Flip this single flag to go from a quick test run to full processing
# of the ~43,000+ frame sequence -- none of the processing code below
# needs to change either way.
# ------------------------------------------------------------------
TEST_MODE  = True
MAX_FRAMES = 10          # only used when TEST_MODE is True

def discover_frame_ids(velodyne_dir, label_dir, has_labels):
    """Find frame IDs to process.

    If has_labels is True (sequences 00-10): only frames that have BOTH
    a .bin and a .label file are used, exactly as before.

    If has_labels is False (sequences 11-21, no released ground truth):
    every frame with a .bin file is used -- there is nothing to pair
    against, and label-dependent stages are skipped entirely later on.
    """
    bin_ids = {
        os.path.splitext(os.path.basename(p))[0]
        for p in glob.glob(os.path.join(velodyne_dir, "*.bin"))
    }

    if not has_labels:
        return sorted(bin_ids), [], []

    label_ids = {
        os.path.splitext(os.path.basename(p))[0]
        for p in glob.glob(os.path.join(label_dir, "*.label"))
    }

    matched        = sorted(bin_ids & label_ids)
    missing_labels = sorted(bin_ids - label_ids)   # has .bin, no .label
    missing_points = sorted(label_ids - bin_ids)   # has .label, no .bin
    return matched, missing_labels, missing_points

ALL_FRAME_IDS, MISSING_LABELS, MISSING_POINTS = discover_frame_ids(
    VELODYNE_DIR, LABEL_DIR, HAS_LABELS
)

print("=" * 70)
print("FRAME DISCOVERY")
print("=" * 70)

if HAS_LABELS:
    print(f"  Matched frames (.bin + .label both present) : {len(ALL_FRAME_IDS):,}")
    print(f"  .bin with no matching .label                : {len(MISSING_LABELS):,}")
    print(f"  .label with no matching .bin                : {len(MISSING_POINTS):,}")
    if MISSING_LABELS:
        preview = ", ".join(MISSING_LABELS[:10])
        print(f"    e.g. missing labels for: {preview}"
              f"{' ...' if len(MISSING_LABELS) > 10 else ''}")
    if MISSING_POINTS:
        preview = ", ".join(MISSING_POINTS[:10])
        print(f"    e.g. missing points for: {preview}"
              f"{' ...' if len(MISSING_POINTS) > 10 else ''}")
else:
    print(f"  No labels for this sequence -- using every .bin frame          : "
          f"{len(ALL_FRAME_IDS):,}")
    print("  Voxelization + 2.5D projection will run on all of them;")
    print("  label remap / propagation / sparse label save will be skipped.")

if not ALL_FRAME_IDS:
    raise RuntimeError(
        "No frames found to process. Double-check POINTCLOUD_ROOT / "
        "LABEL_ROOT / SEQUENCE above."
    )

if TEST_MODE:
    FRAME_IDS = ALL_FRAME_IDS[:MAX_FRAMES]
    print(f"\n  TEST_MODE=True  -> processing {len(FRAME_IDS)} of "
          f"{len(ALL_FRAME_IDS):,} frames")
else:
    FRAME_IDS = ALL_FRAME_IDS
    print(f"\n  TEST_MODE=False -> processing all {len(FRAME_IDS):,} frames")

print(f"  First frame ID  : {FRAME_IDS[0]}")
print(f"  Last  frame ID  : {FRAME_IDS[-1]}")


FRAME DISCOVERY
  Matched frames (.bin + .label both present) : 4,541
  .bin with no matching .label                : 0
  .label with no matching .bin                : 0

  TEST_MODE=True  -> processing 10 of 4,541 frames
  First frame ID  : 000000
  Last  frame ID  : 000009


In [43]:
# ============================================================
# 5. POINT-CLOUD LOADING
# ============================================================
def load_points(frame_id):
    """Load one SemanticKITTI point cloud (N,4: x, y, z, intensity)."""
    path = os.path.join(VELODYNE_DIR, f"{frame_id}.bin")
    return np.fromfile(path, dtype=np.float32).reshape(-1, 4)


# ── Quick sanity check / exploration on the first frame only ──
_fid    = FRAME_IDS[0]
_points = load_points(_fid)
_dist   = np.sqrt(_points[:, 0] ** 2 + _points[:, 1] ** 2)

print("=" * 55)
print(f"DATA EXPLORATION \u2014 frame {_fid}")
print("=" * 55)
print(f"\n  Shape          : {_points.shape}")
print(f"  Points         : {len(_points):,}")
print(f"\n  X range        : {_points[:,0].min():.2f} to {_points[:,0].max():.2f} m")
print(f"  Y range        : {_points[:,1].min():.2f} to {_points[:,1].max():.2f} m")
print(f"  Z range        : {_points[:,2].min():.2f} to {_points[:,2].max():.2f} m")
print(f"  Intensity      : {_points[:,3].min():.2f} to {_points[:,3].max():.2f}")
print(f"  NaN count      : {np.isnan(_points).sum()}")
print(f"\n  Horizontal dist:")
print(f"    Min          : {_dist.min():.2f} m")
print(f"    Max          : {_dist.max():.2f} m")

print(f"\n  Distance percentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"    {p:3d}%       : {np.percentile(_dist, p):.2f} m")

for _label, _lo, _hi in [("0\u201310m", 0, 10), ("10\u201330m", 10, 30), ("30\u2013100m", 30, 100)]:
    _mask = (_dist >= _lo) & (_dist < _hi)
    print(f"\n  {_label}: {_mask.sum():,} points ({_mask.mean()*100:.1f}%)")

del _points, _dist  # exploration only -- do not keep a whole frame around


DATA EXPLORATION — frame 000000

  Shape          : (124668, 4)
  Points         : 124,668

  X range        : -78.09 to 77.97 m
  Y range        : -55.72 to 44.88 m
  Z range        : -11.56 to 2.83 m
  Intensity      : 0.00 to 0.99
  NaN count      : 0

  Horizontal dist:
    Min          : 1.25 m
    Max          : 79.74 m

  Distance percentiles:
     25%       : 6.59 m
     50%       : 9.99 m
     75%       : 15.81 m
     90%       : 26.04 m
     95%       : 37.78 m
     99%       : 56.62 m

  0–10m: 62,364 points (50.0%)

  10–30m: 52,932 points (42.5%)

  30–100m: 9,372 points (7.5%)


In [44]:
# ============================================================
# 6. ADAPTIVE VOXELIZATION
# ============================================================
SIZE_MAP = np.array([0.05, 0.15, 0.50], dtype=np.float32)

BANDS = [
    (0,  10,  0.05, 0),
    (10, 30,  0.15, 1),
    (30, 100, 0.50, 2),
]

# Bit-packed key layout: [level:2][cx:19][cy:19][cz:19] -> fits in int64
COORD_BITS   = 19
COORD_OFFSET = 1 << (COORD_BITS - 1)   # 262144; real coords never get close to this
COORD_MASK   = (1 << COORD_BITS) - 1


# ── PIPELINE CLASS (algorithm unchanged from the original) ────
class VoxelPipeline:
    """
    Adaptive voxelization with pre-allocated buffers.
    Processes one LiDAR frame at a time.

    Optimizations vs. the original:
      1. Single packed int64 voxel key instead of a 4xint32-as-void view
         -> np.unique sorts plain int64s, which is faster than sorting
         16-byte structured records.
      2. np.add.at / np.minimum.at / np.maximum.at replaced with
         np.bincount (sum/count) + one sort + reduceat (min/max).
         np.add.at and friends are documented as unbuffered, un-vectorized
         ufuncs -- for a frame with tens of thousands of voxels they are
         usually the single biggest cost in a pipeline like this, well
         above the cost of np.unique itself.
      3. Auto-grows its voxel buffers instead of throwing an IndexError
         if a frame produces more voxels than max_voxels expected.

    Output feature dict has exactly the same keys/shapes as before, so
    save_voxels/load_voxels and the main loop below are unchanged.
    """
    def __init__(self,
                 max_points=150000,
                 max_voxels=100000):
        self.max_points = max_points
        self.max_voxels = max_voxels

        self.dist     = np.zeros(max_points, dtype=np.float32)
        self.levels   = np.zeros(max_points, dtype=np.int32)
        self.vox_size = np.zeros(max_points, dtype=np.float32)

    def _grow_points(self, N):
        if N > self.max_points:
            self.max_points = N
            self.dist = np.zeros(N, dtype=np.float32)
            self.levels = np.zeros(N, dtype=np.int32)
            self.vox_size = np.zeros(N, dtype=np.float32)

    def process(self, points):
        N = len(points)
        self._grow_points(N)

        # ── Step 1: assign resolution level per point ──
        dist     = self.dist[:N]
        levels   = self.levels[:N]
        vox_size = self.vox_size[:N]

        np.sqrt(points[:,0]**2 + points[:,1]**2, out=dist)

        levels[:]   = 2;  vox_size[:] = 0.50   # default: far
        m1 = dist < 30;   levels[m1]  = 1; vox_size[m1] = 0.15
        m0 = dist < 10;   levels[m0]  = 0; vox_size[m0] = 0.05

        # ── Step 2: compute voxel coordinates ─────────
        coords = np.floor(
            points[:,:3] / vox_size[:,None]
        ).astype(np.int64)

        # ── Step 3: single packed int64 key (replaces void-view key) ──
        cx = coords[:,0] + COORD_OFFSET
        cy = coords[:,1] + COORD_OFFSET
        cz = coords[:,2] + COORD_OFFSET
        keys = (levels.astype(np.int64) << (3 * COORD_BITS)) \
             | (cx << (2 * COORD_BITS)) \
             | (cy << COORD_BITS) \
             | cz

        unique_packed, inverse = np.unique(keys, return_inverse=True)
        V = unique_packed.shape[0]

        # unpack back to (level, cx, cy, cz) int32 -- same shape as before
        ucz = (unique_packed & COORD_MASK) - COORD_OFFSET
        ucy = ((unique_packed >> COORD_BITS) & COORD_MASK) - COORD_OFFSET
        ucx = ((unique_packed >> (2 * COORD_BITS)) & COORD_MASK) - COORD_OFFSET
        ulv = (unique_packed >> (3 * COORD_BITS))
        unique_keys = np.stack(
            [ulv, ucx, ucy, ucz], axis=1
        ).astype(np.int32)

        # ── Step 4: accumulate voxel features (bincount + reduceat) ──
        z     = points[:,2].astype(np.float64)
        inten = points[:,3].astype(np.float64)

        count  = np.bincount(inverse, minlength=V).astype(np.int32)
        z_sum  = np.bincount(inverse, weights=z, minlength=V)
        z_sum2 = np.bincount(inverse, weights=z**2, minlength=V)
        i_sum  = np.bincount(inverse, weights=inten, minlength=V)

        # min/max can't be done with bincount -> one sort + reduceat,
        # still far cheaper than np.minimum.at/np.maximum.at scatter
        order       = np.argsort(inverse, kind='stable')
        sorted_inv  = inverse[order]
        sorted_z    = z[order]
        first_idx   = np.searchsorted(sorted_inv, np.arange(V))
        z_min = np.minimum.reduceat(sorted_z, first_idx)
        z_max = np.maximum.reduceat(sorted_z, first_idx)

        z_mean  = z_sum  / count
        z_var   = z_sum2 / count - z_mean**2
        z_range = z_max  - z_min
        i_mean  = i_sum  / count

        # ── Step 5: voxel centers ─────────────────────
        lv = unique_keys[:,0]
        vs = SIZE_MAP[lv]
        cxf = (unique_keys[:,1] + 0.5) * vs
        cyf = (unique_keys[:,2] + 0.5) * vs
        czf = (unique_keys[:,3] + 0.5) * vs

        features = {
            "unique_keys":    unique_keys,
            "levels":         lv,
            "vox_size":       vs,
            "center_x":       cxf,
            "center_y":       cyf,
            "center_z":       czf,
            "point_count":    count,
            "height_mean":    z_mean,
            "height_min":     z_min,
            "height_max":     z_max,
            "height_var":     z_var,
            "height_range":   z_range,
            "intensity_mean": i_mean,
        }

        return features, inverse.astype(np.int32)


# ── SAVE (now under VOXEL_DIR = /kaggle/working/voxel_store) ──
def save_voxels(features, inverse, frame_id):
    center_xyz = np.stack([
        features["center_x"],
        features["center_y"],
        features["center_z"]
    ], axis=1).astype(np.float32)

    np.savez_compressed(
        os.path.join(VOXEL_DIR, f"{frame_id}_voxels.npz"),
        keys         = features["unique_keys"].astype(np.int32),
        levels       = features["levels"].astype(np.int32),
        vox_size     = features["vox_size"].astype(np.float32),
        center_xyz   = center_xyz,
        point_count  = features["point_count"].astype(np.int32),
        height_mean  = features["height_mean"].astype(np.float32),
        height_min   = features["height_min"].astype(np.float32),
        height_max   = features["height_max"].astype(np.float32),
        height_var   = features["height_var"].astype(np.float32),
        height_range = features["height_range"].astype(np.float32),
        intensity    = features["intensity_mean"].astype(np.float32),
        inverse      = inverse,
    )


# ── LOAD ─────────────────────────────────────────────────
def load_voxels(frame_id):
    d = np.load(
        os.path.join(VOXEL_DIR, f"{frame_id}_voxels.npz")
    )
    return {
        "unique_keys": d["keys"],
        "levels":      d["levels"],
        "vox_size":    d["vox_size"],
        "center_x":    d["center_xyz"][:,0],
        "center_y":    d["center_xyz"][:,1],
        "center_z":    d["center_xyz"][:,2],
        "point_count": d["point_count"],
        "height_mean": d["height_mean"],
        "height_min":  d["height_min"],
        "height_max":  d["height_max"],
        "height_var":  d["height_var"],
        "height_range":d["height_range"],
        "intensity_mean": d["intensity"],
        "inverse":     d["inverse"],
    }


# ── UNIFORM BASELINE (for benchmark) ─────────────────────
def uniform_voxel_count(points, size=0.05):
    coords = np.floor(
        points[:,:3] / size
    ).astype(np.int32)
    keys   = np.ascontiguousarray(coords).view(
        np.dtype((np.void, 12))
    ).ravel()
    return len(np.unique(keys))


# ── MAIN LOOP (frame-by-frame, one frame in RAM at a time) ──
print("=" * 80)
print(f"ADAPTIVE VOXELIZATION \u2014 {len(FRAME_IDS)} FRAMES")
print("=" * 80)

pipeline = VoxelPipeline(max_points=150000, max_voxels=100000)
pipeline.process(load_points(FRAME_IDS[0]))  # warmup

print(f"\n{'Frame':<12} {'Points':>8} {'Uniform':>10} "
      f"{'Adaptive':>10} {'Reduction':>10} "
      f"{'L0':>7} {'L1':>7} {'L2':>7} "
      f"{'ms':>8} {'FPS':>6}")
print("-" * 95)

voxel_results       = []
VOXELIZATION_FAILED = []

for fid in FRAME_IDS:
    try:
        points = load_points(fid)
        unif   = uniform_voxel_count(points)

        t0 = time.perf_counter()
        features, inv = pipeline.process(points)
        elapsed = (time.perf_counter() - t0) * 1000

        save_voxels(features, inv, fid)

        lv_counts = Counter(features["levels"].tolist())
        adap = len(features["unique_keys"])

        voxel_results.append({
            "frame": fid, "points": len(points), "uniform": unif, "adaptive": adap,
            "reduction": unif / adap, "l0": lv_counts[0], "l1": lv_counts[1],
            "l2": lv_counts[2], "ms": elapsed,
        })

        print(f"{fid:<12} {len(points):>8,} {unif:>10,} "
              f"{adap:>10,} {unif/adap:>9.2f}x "
              f"{lv_counts[0]:>7,} {lv_counts[1]:>7,} "
              f"{lv_counts[2]:>7,} {elapsed:>7.1f}ms "
              f"{1000/elapsed:>5.1f}")

    except Exception as e:
        VOXELIZATION_FAILED.append((fid, str(e)))
        print(f"{fid:<12}  \u2717 FAILED: {e}")

def _avg(k, rows):
    return float(np.mean([r[k] for r in rows])) if rows else float("nan")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
if voxel_results:
    print(f"  Avg points     : {_avg('points', voxel_results):,.0f}")
    print(f"  Avg uniform    : {_avg('uniform', voxel_results):,.0f}")
    print(f"  Avg adaptive   : {_avg('adaptive', voxel_results):,.0f}")
    print(f"  Avg reduction  : {_avg('reduction', voxel_results):.2f}x")
    print(f"  Avg time       : {_avg('ms', voxel_results):.1f} ms")
    print(f"  Avg FPS        : {1000/_avg('ms', voxel_results):.1f}")
    print(f"  Avg L0 (5cm)   : {_avg('l0', voxel_results):,.0f} "
          f"({_avg('l0', voxel_results)/_avg('adaptive', voxel_results)*100:.1f}%)")
    print(f"  Avg L1 (15cm)  : {_avg('l1', voxel_results):,.0f} "
          f"({_avg('l1', voxel_results)/_avg('adaptive', voxel_results)*100:.1f}%)")
    print(f"  Avg L2 (50cm)  : {_avg('l2', voxel_results):,.0f} "
          f"({_avg('l2', voxel_results)/_avg('adaptive', voxel_results)*100:.1f}%)")
print(f"  Failed frames  : {len(VOXELIZATION_FAILED)}")
print(f"\n  Voxel files saved to: {VOXEL_DIR}/")


ADAPTIVE VOXELIZATION — 10 FRAMES

Frame          Points    Uniform   Adaptive  Reduction      L0      L1      L2       ms    FPS
-----------------------------------------------------------------------------------------------
000000        124,668     91,767     62,158      1.48x  33,922  23,934   4,302    44.7ms  22.4
000001        124,605     91,495     62,256      1.47x  34,389  23,592   4,275    38.1ms  26.3
000002        124,478     91,177     62,134      1.47x  34,635  23,373   4,126    32.1ms  31.2
000003        124,167     90,572     61,926      1.46x  35,151  22,772   4,003    40.6ms  24.6
000004        123,969     90,316     61,899      1.46x  35,461  22,426   4,012    32.2ms  31.0
000005        123,924     89,992     61,519      1.46x  35,126  22,359   4,034    45.1ms  22.2
000006        123,373     89,469     60,909      1.47x  34,838  22,167   3,904    31.0ms  32.3
000007        122,765     89,219     60,534      1.47x  34,671  21,969   3,894    32.5ms  30.8
000008        

In [45]:
# ============================================================
# 7. 2.5D PROJECTION
# ============================================================
# Feature channel indices
CH = {
    "height_max":   0,
    "height_min":   1,
    "height_range": 2,
    "height_mean":  3,
    "point_count":  4,
    "voxel_count":  5,
    "intensity":    6,
    "is_occupied":  7,
    "pts_per_m2":   8,
}
N_CH = len(CH)


# ── PROJECTION (algorithm unchanged) ──────────────────────
def project_level(features, level):
    mask = features["levels"] == level
    if mask.sum() == 0:
        return None, None

    vs     = float(SIZE_MAP[level])
    cx     = features["center_x"][mask]
    cy     = features["center_y"][mask]

    ix     = np.floor(cx / vs).astype(np.int32)
    iy     = np.floor(cy / vs).astype(np.int32)
    ix_min = ix.min();  iy_min = iy.min()
    ix_rel = ix - ix_min
    iy_rel = iy - iy_min
    H      = int(ix_rel.max()) + 1
    W      = int(iy_rel.max()) + 1
    flat   = ix_rel * W + iy_rel
    C      = H * W

    h_max  = features["height_max"][mask].astype(np.float32)
    h_min  = features["height_min"][mask].astype(np.float32)
    h_mean = features["height_mean"][mask].astype(np.float32)
    pcount = features["point_count"][mask].astype(np.float32)
    inten  = features["intensity_mean"][mask].astype(np.float32)

    g_hmax    = np.full(C, -np.inf, dtype=np.float32)
    g_hmin    = np.full(C,  np.inf, dtype=np.float32)
    g_pcount  = np.zeros(C, dtype=np.float32)
    g_vcount  = np.zeros(C, dtype=np.float32)
    g_hmean_w = np.zeros(C, dtype=np.float32)
    g_inten_w = np.zeros(C, dtype=np.float32)

    np.maximum.at(g_hmax,    flat, h_max)
    np.minimum.at(g_hmin,    flat, h_min)
    np.add.at(g_pcount,      flat, pcount)
    np.add.at(g_vcount,      flat, 1)
    np.add.at(g_hmean_w,     flat, h_mean * pcount)
    np.add.at(g_inten_w,     flat, inten  * pcount)

    occ      = g_vcount > 0
    g_hmean  = np.zeros(C, dtype=np.float32)
    g_inten  = np.zeros(C, dtype=np.float32)
    g_hrange = np.zeros(C, dtype=np.float32)

    g_hmean[occ]  = g_hmean_w[occ] / g_pcount[occ]
    g_inten[occ]  = g_inten_w[occ] / g_pcount[occ]
    g_hrange[occ] = g_hmax[occ] - g_hmin[occ]
    g_hmax[~occ]  = 0.0
    g_hmin[~occ]  = 0.0

    grid = np.zeros((H, W, N_CH), dtype=np.float32)
    grid[:,:,CH["height_max"]]   = g_hmax.reshape(H, W)
    grid[:,:,CH["height_min"]]   = g_hmin.reshape(H, W)
    grid[:,:,CH["height_range"]] = g_hrange.reshape(H, W)
    grid[:,:,CH["height_mean"]]  = g_hmean.reshape(H, W)
    grid[:,:,CH["point_count"]]  = g_pcount.reshape(H, W)
    grid[:,:,CH["voxel_count"]]  = g_vcount.reshape(H, W)
    grid[:,:,CH["intensity"]]    = g_inten.reshape(H, W)
    grid[:,:,CH["is_occupied"]]  = occ.reshape(H,W).astype(np.float32)
    grid[:,:,CH["pts_per_m2"]]   = (g_pcount / vs**2).reshape(H, W)

    meta = {
        "H":        H,
        "W":        W,
        "vox_size": vs,
        "ix_min":   int(ix_min),
        "iy_min":   int(iy_min),
        "occupied": int(occ.sum()),
    }
    return grid, meta


def project_2d(features):
    grids = {};  metas = {}
    for lv in [0, 1, 2]:
        grid, meta = project_level(features, lv)
        if grid is not None:
            grids[lv] = grid
            metas[lv] = meta
    return grids, metas


def save_map(grids, metas, frame_id):
    meta_arr = np.array([
        [metas[lv]["H"],
         metas[lv]["W"],
         metas[lv]["ix_min"],
         metas[lv]["iy_min"]]
        for lv in [0, 1, 2]
    ], dtype=np.int32)

    np.savez_compressed(
        os.path.join(MAP_DIR, f"{frame_id}_map2d.npz"),
        L0_5cm     = grids[0],
        L1_15cm    = grids[1],
        L2_50cm    = grids[2],
        meta_hwoff = meta_arr,
    )


def load_map(frame_id):
    d        = np.load(
        os.path.join(MAP_DIR, f"{frame_id}_map2d.npz")
    )
    meta_arr = d["meta_hwoff"]
    grids    = {
        0: d["L0_5cm"],
        1: d["L1_15cm"],
        2: d["L2_50cm"],
    }
    metas = {}
    for i, lv in enumerate([0, 1, 2]):
        H, W, ix_min, iy_min = meta_arr[i]
        metas[lv] = {
            "H":        int(H),
            "W":        int(W),
            "ix_min":   int(ix_min),
            "iy_min":   int(iy_min),
            "vox_size": float(SIZE_MAP[lv]),
        }
    return grids, metas


# ── MAIN LOOP ─────────────────────────────────────────────
print("=" * 70)
print(f"2.5D MAP PROJECTION \u2014 {len(FRAME_IDS)} FRAMES")
print("=" * 70)

_failed_voxel_ids  = {fid for fid, _ in VOXELIZATION_FAILED}
PROJECTION_FAILED  = []
all_proj_ms        = []

print(f"\n{'Frame':<12} {'L0 grid':>14} {'L1 grid':>14} "
      f"{'L2 grid':>14} {'ms':>8} {'FPS':>6}")
print("-" * 75)

for fid in FRAME_IDS:
    if fid in _failed_voxel_ids:
        PROJECTION_FAILED.append((fid, "skipped: voxelization failed"))
        continue
    try:
        features = load_voxels(fid)

        t0           = time.perf_counter()
        grids, metas = project_2d(features)
        elapsed      = (time.perf_counter() - t0) * 1000
        all_proj_ms.append(elapsed)

        save_map(grids, metas, fid)

        l0 = metas[0];  l1 = metas[1];  l2 = metas[2]
        print(f"{fid:<12} "
              f"{l0['H']}\u00d7{l0['W']:>5} "
              f"{l1['H']}\u00d7{l1['W']:>5} "
              f"{l2['H']}\u00d7{l2['W']:>5} "
              f"{elapsed:>8.1f}ms "
              f"{1000/elapsed:>5.1f}")
    except Exception as e:
        PROJECTION_FAILED.append((fid, str(e)))
        print(f"{fid:<12}  \u2717 FAILED: {e}")

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
if all_proj_ms:
    print(f"  Avg projection time : {np.mean(all_proj_ms):.1f} ms")
    print(f"  Avg FPS             : {1000/np.mean(all_proj_ms):.1f}")
print(f"  Failed frames       : {len(PROJECTION_FAILED)}")

_proj_failed_ids = {fid for fid, _ in PROJECTION_FAILED}
_first_ok = next((fid for fid in FRAME_IDS
                   if fid not in _failed_voxel_ids and fid not in _proj_failed_ids), None)
if _first_ok:
    print(f"\n  Occupancy (frame {_first_ok}):")
    grids_0, metas_0 = load_map(_first_ok)
    for lv in [0, 1, 2]:
        g   = grids_0[lv]
        occ = g[:,:,CH["is_occupied"]].astype(bool).sum()
        tot = metas_0[lv]["H"] * metas_0[lv]["W"]
        print(f"    L{lv}: {occ:,} / {tot:,} cells "
              f"({occ/tot*100:.1f}% occupied)")
print(f"\n  Maps saved to: {MAP_DIR}/")


2.5D MAP PROJECTION — 10 FRAMES

Frame               L0 grid        L1 grid        L2 grid       ms    FPS
---------------------------------------------------------------------------
000000       397×  399 398×  342 313×  202     39.3ms  25.5
000001       399×  400 399×  332 319×  213     37.3ms  26.8
000002       400×  400 400×  324 317×  196     42.6ms  23.5
000003       399×  400 398×  317 318×  181     34.6ms  28.9
000004       398×  396 398×  309 317×  176     33.7ms  29.6
000005       396×  396 398×  302 313×  172     30.7ms  32.6
000006       391×  393 399×  296 313×  171     31.5ms  31.8
000007       395×  389 399×  286 317×  170     32.2ms  31.0
000008       400×  388 400×  279 316×  169     29.4ms  34.0
000009       398×  390 400×  274 315×  169     29.9ms  33.5

SUMMARY
  Avg projection time : 34.1 ms
  Avg FPS             : 29.3
  Failed frames       : 0

  Occupancy (frame 000000):
    L0: 25,455 / 158,403 cells (16.1% occupied)
    L1: 13,358 / 136,116 cells (9.8% occupie

In [46]:
# ============================================================
# 8. LABEL LOADING / REMAPPING
# ============================================================
N_CLASSES = 5
IGNORE    = 0

CLASS_NAMES = {
    0: "ignore",
    1: "drivable",
    2: "non_drivable_terrain",
    3: "static_obstacle",
    4: "dynamic_object",
}

# ── LABEL LOOKUP TABLE (unchanged) ────────────────────────
RAW_LABELS = {
    0:  "unlabeled",     1:  "outlier",
    10: "car",           11: "bicycle",
    13: "bus",           15: "motorcycle",
    16: "on-rails",      18: "truck",
    20: "other-vehicle",
    30: "person",        31: "bicyclist",
    32: "motorcyclist",
    40: "road",          44: "parking",
    48: "sidewalk",      49: "other-ground",
    50: "building",      51: "fence",
    52: "other-structure",
    60: "lane-marking",
    70: "vegetation",    71: "trunk",
    72: "terrain",       80: "pole",
    81: "traffic-sign",  99: "other-object",
    252: "moving-car",         253: "moving-bicyclist",
    254: "moving-person",      255: "moving-motorcyclist",
    256: "moving-on-rails",    257: "moving-bus",
    258: "moving-truck",       259: "moving-other-vehicle",
}

NAME_TO_CLASS = {
    "unlabeled": 0,    "outlier": 0,
    "other-object": 0, "other-structure": 3,
    "road": 1,         "parking": 1,
    "lane-marking": 1,
    "sidewalk": 2,     "terrain": 2,
    "other-ground": 2,
    "building": 3,     "fence": 3,
    "pole": 3,         "traffic-sign": 3,
    "vegetation": 3,   "trunk": 3,
    "car": 4,          "bicycle": 4,
    "bus": 4,          "motorcycle": 4,
    "on-rails": 4,     "truck": 4,
    "other-vehicle": 4,"person": 4,
    "bicyclist": 4,    "motorcyclist": 4,
    "moving-car": 4,          "moving-bicyclist": 4,
    "moving-person": 4,       "moving-motorcyclist": 4,
    "moving-on-rails": 4,     "moving-bus": 4,
    "moving-truck": 4,        "moving-other-vehicle": 4,
}

_MAX_ID = max(RAW_LABELS.keys()) + 1
LUT     = np.zeros(_MAX_ID, dtype=np.uint8)
for raw_id, name in RAW_LABELS.items():
    LUT[raw_id] = NAME_TO_CLASS[name]


def remap_labels(frame_id):
    """Load + remap one SemanticKITTI .label file -- READ FROM LABEL_DIR
    (dataset 2), never from the point-cloud dataset."""
    path    = os.path.join(LABEL_DIR, f"{frame_id}.label")
    raw     = np.fromfile(path, dtype=np.uint32)
    raw_sem = (raw & 0xFFFF).astype(np.int32)
    return LUT[np.clip(raw_sem, 0, _MAX_ID - 1)]

print("Label LUT built:", LUT.shape[0], "raw IDs ->", N_CLASSES, "classes")


Label LUT built: 260 raw IDs -> 5 classes


In [47]:
# ============================================================
# 9. LABEL PROPAGATION
# ============================================================
def assign_voxel_labels(sem_label, features):
    """Majority vote per voxel using stored inverse."""
    inverse   = features["inverse"]
    V         = len(features["unique_keys"])
    combined  = (inverse * N_CLASSES
                 + sem_label.astype(np.int32))
    flat_hist = np.bincount(
        combined, minlength=V * N_CLASSES
    ).reshape(V, N_CLASSES).astype(np.float32)

    voxel_class = np.argmax(
        flat_hist, axis=1
    ).astype(np.uint8)
    total       = flat_hist.sum(axis=1)
    voxel_conf  = (
        flat_hist[
            np.arange(V),
            voxel_class.astype(np.int32)
        ] / np.maximum(total, 1)
    ).astype(np.float32)

    return voxel_class, voxel_conf


def assign_cell_labels(features, voxel_class, metas):
    """Project voxel labels into 2D cells."""
    levels_arr = features["levels"]
    pcount_arr = features["point_count"].astype(np.float32)
    cx_all     = features["center_x"]
    cy_all     = features["center_y"]
    cell_labels = {};  cell_conf = {}

    for lv in [0, 1, 2]:
        meta   = metas[lv]
        H      = meta["H"];  W = meta["W"]
        vs     = meta["vox_size"]
        ix_min = meta["ix_min"]
        iy_min = meta["iy_min"]

        mask = levels_arr == lv
        if mask.sum() == 0:
            cell_labels[lv] = np.zeros((H,W), dtype=np.uint8)
            cell_conf[lv]   = np.zeros((H,W), dtype=np.float32)
            continue

        cx = cx_all[mask];  cy = cy_all[mask]
        vc = voxel_class[mask].astype(np.int32)
        pc = pcount_arr[mask]

        ix_rel = (np.floor(cx/vs).astype(np.int32) - ix_min)
        iy_rel = (np.floor(cy/vs).astype(np.int32) - iy_min)
        valid  = ((ix_rel>=0)&(ix_rel<H)&
                  (iy_rel>=0)&(iy_rel<W))

        ix_rel = ix_rel[valid];  iy_rel = iy_rel[valid]
        vc     = vc[valid];      pc     = pc[valid]

        flat      = ix_rel * W + iy_rel
        combined  = (flat * N_CLASSES + vc).astype(np.int64)
        flat_hist = np.bincount(
            combined,
            weights   = pc,
            minlength = H * W * N_CLASSES
        ).reshape(H*W, N_CLASSES).astype(np.float32)

        cls_flat  = np.argmax(
            flat_hist, axis=1
        ).astype(np.uint8)
        tot_flat  = flat_hist.sum(axis=1)
        occ       = tot_flat > 0
        conf_flat = np.zeros(H*W, dtype=np.float32)
        conf_flat[occ] = (
            flat_hist[occ, cls_flat[occ].astype(np.int32)]
            / tot_flat[occ]
        )
        cell_labels[lv] = cls_flat.reshape(H, W)
        cell_conf[lv]   = conf_flat.reshape(H, W)

    return cell_labels, cell_conf

print("Label propagation functions ready: assign_voxel_labels, assign_cell_labels")


Label propagation functions ready: assign_voxel_labels, assign_cell_labels


In [48]:
# ============================================================
# 10. OUTPUT / RESULT HANDLING
# ============================================================
def save_sparse(frame_id, grids, metas,
                cell_labels, cell_conf):
    counts = {}
    for lv in [0, 1, 2]:
        grid     = grids[lv]
        labels   = cell_labels[lv]
        conf     = cell_conf[lv]
        occupied = grid[:,:,CH["is_occupied"]].astype(bool)
        valid    = occupied & (labels != IGNORE)
        counts[lv] = int(valid.sum())

        if valid.sum() == 0:
            continue

        ix, iy = np.where(valid)
        base   = os.path.join(
            SPARSE_DIR, f"{frame_id}_l{lv}"
        )
        np.save(base + "_coords.npy",
                np.stack([ix,iy],axis=1).astype(np.int32))
        np.save(base + "_features.npy",
                grid[ix,iy,:].astype(np.float32))
        np.save(base + "_labels.npy",
                labels[ix,iy].astype(np.uint8))
        np.save(base + "_confidence.npy",
                conf[ix,iy].astype(np.float32))
    return counts


def verify_sparse(frame_id):
    errors = []
    for lv in [0, 1, 2]:
        base = os.path.join(
            SPARSE_DIR, f"{frame_id}_l{lv}"
        )
        paths = {
            k: base + f"_{k}.npy"
            for k in ["coords","features",
                      "labels","confidence"]
        }
        if not all(os.path.exists(p)
                   for p in paths.values()):
            errors.append(f"L{lv} files missing")
            continue

        coords = np.load(paths["coords"])
        feats  = np.load(paths["features"])
        labels = np.load(paths["labels"])
        conf   = np.load(paths["confidence"])
        N      = len(coords)

        if feats.shape  != (N, 9):
            errors.append(f"L{lv} feat shape {feats.shape}")
        if labels.shape != (N,):
            errors.append(f"L{lv} label shape")
        if (labels == 0).any():
            errors.append(f"L{lv} ignore labels present")
        if np.isnan(feats).any():
            errors.append(f"L{lv} NaN in features")

    return errors


# ── result containers (always defined, even when labels are skipped,
#    so the final summary cell can reference them safely) ──
label_prop_results = []
LABEL_PROP_FAILED  = []
class_totals        = np.zeros(N_CLASSES, dtype=np.int64)
t_lbl_acc  = [];  t_vox_acc  = []
t_cell_acc = [];  t_save_acc = []

_skip_ids = _failed_voxel_ids | _proj_failed_ids

if not HAS_LABELS:
    # ------------------------------------------------------------
    # No ground truth for this sequence (e.g. SemanticKITTI 11-21).
    # Voxelization + 2.5D projection already ran on every frame in
    # the earlier cells and their outputs are sitting in VOXEL_DIR /
    # MAP_DIR. There is nothing to remap/propagate/save here, so we
    # skip this stage cleanly rather than fail on a missing .label file.
    # ------------------------------------------------------------
    print("=" * 70)
    print("LABEL PROPAGATION SKIPPED -- no labels for this sequence")
    print("=" * 70)
    print(f"  SEQUENCE='{SEQUENCE}' has no ground-truth labels (HAS_LABELS=False).")
    print(f"  Voxel features : {VOXEL_DIR}/  ({len(FRAME_IDS) - len(_failed_voxel_ids)} frames)")
    print(f"  2.5D maps      : {MAP_DIR}/  ({len(FRAME_IDS) - len(_skip_ids)} frames)")
    print(f"  {SPARSE_DIR}/ will remain empty for this run -- there is no")
    print("  label-propagated output to produce without ground truth.")
else:
    # ── MAIN LOOP ─────────────────────────────────────────────
    print(f"\n{'Frame':<10} {'L0':>7} {'L1':>7} {'L2':>7} "
          f"{'Lbl ms':>8} {'Vox ms':>8} {'Cell ms':>8} "
          f"{'Save ms':>8} {'Total ms':>10} {'FPS':>6}  Check")
    print("-" * 100)

    for fid in FRAME_IDS:
        if fid in _skip_ids:
            LABEL_PROP_FAILED.append((fid, "skipped: earlier stage failed"))
            continue

        try:
            t_full = time.perf_counter()

            # Load everything for this one frame only
            features     = load_voxels(fid)
            grids, metas = load_map(fid)

            # Remap labels (from LABEL_DIR, dataset 2)
            t0    = time.perf_counter()
            sem_5 = remap_labels(fid)
            t_lbl = (time.perf_counter() - t0) * 1000
            t_lbl_acc.append(t_lbl)

            # Voxel labels
            t0 = time.perf_counter()
            voxel_class, voxel_conf = assign_voxel_labels(sem_5, features)
            t_vox = (time.perf_counter() - t0) * 1000
            t_vox_acc.append(t_vox)

            # Cell labels
            t0 = time.perf_counter()
            cell_labels, cell_conf = assign_cell_labels(features, voxel_class, metas)
            t_cell = (time.perf_counter() - t0) * 1000
            t_cell_acc.append(t_cell)

            # Save
            t0     = time.perf_counter()
            counts = save_sparse(fid, grids, metas, cell_labels, cell_conf)
            t_save = (time.perf_counter() - t0) * 1000
            t_save_acc.append(t_save)

            elapsed = (time.perf_counter() - t_full) * 1000
            errors  = verify_sparse(fid)
            check   = "\u2713" if not errors else f"\u2717 {errors[0]}"

            # Class distribution (L1)
            lbl_path = os.path.join(SPARSE_DIR, f"{fid}_l1_labels.npy")
            if os.path.exists(lbl_path):
                lbl = np.load(lbl_path).astype(np.int32)
                for c in range(N_CLASSES):
                    class_totals[c] += (lbl == c).sum()

            label_prop_results.append({
                "frame": fid, "l0": counts[0],
                "l1": counts[1], "l2": counts[2],
                "ms": elapsed,
            })

            print(f"{fid:<10} {counts[0]:>7,} {counts[1]:>7,} "
                  f"{counts[2]:>7,} {t_lbl:>7.1f}ms "
                  f"{t_vox:>7.1f}ms {t_cell:>7.1f}ms "
                  f"{t_save:>7.1f}ms {elapsed:>9.1f}ms "
                  f"{1000/elapsed:>5.1f}  {check}")

        except Exception as e:
            LABEL_PROP_FAILED.append((fid, str(e)))
            print(f"{fid:<10}  \u2717 FAILED: {e}")

    # ── STAGE SUMMARY ─────────────────────────────────────────
    def _avg2(a): return float(np.mean(a)) if a else float("nan")

    print("\n" + "=" * 70)
    print("STAGE TIMING (averages)")
    print("=" * 70)
    print(f"  remap_labels        : {_avg2(t_lbl_acc):>7.1f} ms")
    print(f"  assign_voxel_labels : {_avg2(t_vox_acc):>7.1f} ms")
    print(f"  assign_cell_labels  : {_avg2(t_cell_acc):>7.1f} ms")
    print(f"  save_sparse         : {_avg2(t_save_acc):>7.1f} ms")
    if label_prop_results:
        ms_avg = _avg2([r["ms"] for r in label_prop_results])
        print(f"  \u2500" * 31)
        print(f"  Total avg           : {ms_avg:>7.1f} ms")
        print(f"  Avg FPS             : {1000/ms_avg:>7.1f}")
    print(f"  Failed frames       : {len(LABEL_PROP_FAILED)}")

    print("\n\u2500\u2500 Class distribution (L1) \u2500\u2500")
    total = int(class_totals.sum())
    for c in range(N_CLASSES):
        pct = class_totals[c] / max(total,1) * 100
        bar = "\u2588" * int(pct/100*30)
        print(f"  {CLASS_NAMES[c]:<22} "
              f"{class_totals[c]:>7,} ({pct:5.1f}%)  {bar}")

    print("\n\u2500\u2500 propagated_labels store contents \u2500\u2500")
    npy_files = [f for f in os.listdir(SPARSE_DIR) if f.endswith(".npy")]
    expected = len(label_prop_results) * 3 * 4
    print(f"  .npy files : {len(npy_files)} "
          f"(expected {expected} = "
          f"{len(label_prop_results)} frames \u00d7 3 levels \u00d7 4 arrays)")
    print(f"  Status     : {'\u2713' if len(npy_files)==expected else '\u2717'}")
    total_kb = sum(
        os.path.getsize(os.path.join(SPARSE_DIR,f))/1024
        for f in npy_files
    )
    print(f"  Total size : {total_kb:.1f} KB = {total_kb/1024:.2f} MB")

    # Save metadata into SPARSE_DIR (per-frame detail, same as before)
    meta_out = {
        "frames":      label_prop_results,
        "n_classes":   N_CLASSES,
        "class_names": CLASS_NAMES,
        "feature_channels": CH,
        "class_distribution_l1": {
            CLASS_NAMES[c]: int(class_totals[c])
            for c in range(N_CLASSES)
        },
    }
    with open(os.path.join(SPARSE_DIR, "dataset_meta.json"), "w") as f:
        json.dump(meta_out, f, indent=2)
    print(f"\n  Metadata   : {SPARSE_DIR}/dataset_meta.json")



Frame           L0      L1      L2   Lbl ms   Vox ms  Cell ms  Save ms   Total ms    FPS  Check
----------------------------------------------------------------------------------------------------
000000      25,335  13,239   1,567    19.5ms     4.9ms    27.6ms    11.7ms     154.3ms   6.5  ✓
000001      25,268  13,008   1,543    10.6ms     4.9ms    25.2ms    10.4ms     137.4ms   7.3  ✓
000002      25,007  12,651   1,541    10.4ms     4.7ms    23.9ms     9.8ms     128.3ms   7.8  ✓
000003      24,904  12,310   1,498    10.2ms     4.8ms    23.0ms     9.9ms     130.2ms   7.7  ✓
000004      24,710  12,159   1,549    10.7ms     5.5ms    23.5ms     9.6ms     126.9ms   7.9  ✓
000005      24,545  11,982   1,567    11.4ms     4.6ms    28.1ms    12.0ms     133.8ms   7.5  ✓
000006      24,259  11,646   1,495    11.8ms     4.8ms    24.0ms     9.6ms     140.5ms   7.1  ✓
000007      24,112  11,518   1,489    11.0ms     4.8ms    22.6ms     9.2ms     124.2ms   8.1  ✓
000008      24,190  11,109   1,503

In [49]:
# ============================================================
# 11. FINAL SUMMARY
# ============================================================
def _dir_stats(path, pattern="*"):
    files = glob.glob(os.path.join(path, pattern))
    n     = len(files)
    size_mb = sum(os.path.getsize(f) for f in files) / (1024 * 1024)
    return n, size_mb

n_vox, sz_vox   = _dir_stats(VOXEL_DIR,  "*.npz")
n_map, sz_map   = _dir_stats(MAP_DIR,    "*.npz")
n_sp,  sz_sp    = _dir_stats(SPARSE_DIR, "*.npy")

all_failed = {
    "voxelization": VOXELIZATION_FAILED,
    "projection":   PROJECTION_FAILED,
    "label_propagation": LABEL_PROP_FAILED,
}
n_frames_attempted   = len(FRAME_IDS)
n_voxelized          = len(voxel_results)
n_projected          = len(all_proj_ms)
n_label_propagated   = len(label_prop_results)
n_frames_failed      = len({fid for stage in all_failed.values() for fid, _ in stage})

print("=" * 70)
print("RUN SUMMARY")
print("=" * 70)
print(f"  Sequence                     : {SEQUENCE}")
print(f"  Mode                         : {'TEST' if TEST_MODE else 'FULL'}")
print(f"  Labels available (HAS_LABELS): {HAS_LABELS}")
print(f"  Frames discovered            : {len(ALL_FRAME_IDS):,}")
print(f"  Frames attempted this run    : {n_frames_attempted:,}")
print(f"  Frames voxelized             : {n_voxelized:,}")
print(f"  Frames projected to 2.5D     : {n_projected:,}")
if HAS_LABELS:
    print(f"  Frames label-propagated      : {n_label_propagated:,}")
else:
    print(f"  Frames label-propagated      : 0 (skipped -- no labels for this sequence)")
print(f"  Frames failed (any stage)    : {n_frames_failed:,}")
for stage, failures in all_failed.items():
    if failures:
        print(f"    - {stage}: {len(failures)} "
              f"(e.g. {failures[0][0]}: {failures[0][1]})")

print("\n  Output directories (/kaggle/working/):")
print(f"    {VOXEL_DIR:<45} {n_vox:>5} files, {sz_vox:>8.2f} MB")
print(f"    {MAP_DIR:<45} {n_map:>5} files, {sz_map:>8.2f} MB")
print(f"    {SPARSE_DIR:<45} {n_sp:>5} files, {sz_sp:>8.2f} MB"
      + ("  (empty -- no labels)" if not HAS_LABELS else ""))
print(f"    {RESULTS_DIR:<45}")

# Persist a top-level run summary as well, in RESULTS_DIR
run_summary = {
    "sequence": SEQUENCE,
    "test_mode": TEST_MODE,
    "max_frames": MAX_FRAMES if TEST_MODE else None,
    "has_labels": HAS_LABELS,
    "frames_discovered": len(ALL_FRAME_IDS),
    "frames_attempted": n_frames_attempted,
    "frames_voxelized": n_voxelized,
    "frames_projected": n_projected,
    "frames_label_propagated": n_label_propagated,
    "frames_failed": n_frames_failed,
    "failures": {
        stage: [{"frame": fid, "error": err} for fid, err in failures]
        for stage, failures in all_failed.items()
    },
    "missing_labels_for_bin": MISSING_LABELS[:100],
    "missing_bin_for_labels": MISSING_POINTS[:100],
    "output_dirs": {
        "voxel_store": {"path": VOXEL_DIR, "files": n_vox, "size_mb": round(sz_vox, 3)},
        "maps_2d": {"path": MAP_DIR, "files": n_map, "size_mb": round(sz_map, 3)},
        "propagated_labels": {"path": SPARSE_DIR, "files": n_sp, "size_mb": round(sz_sp, 3)},
    },
}
with open(os.path.join(RESULTS_DIR, "run_summary.json"), "w") as f:
    json.dump(run_summary, f, indent=2)
print(f"\n  Run summary saved to: {os.path.join(RESULTS_DIR, 'run_summary.json')}")

print("\n" + "=" * 70)
print("PERSISTENCE NOTE")
print("=" * 70)
print("  Everything written under /kaggle/working/ (voxel_store/, maps_2d/,")
print("  propagated_labels/, results/) is automatically kept as this")
print("  notebook's output once you run:")
print("      Save Version -> Save & Run All (Commit)")
print("  You do not need to manually export or download these files for")
print("  them to persist -- they will be attached to the saved version")
print("  and can be browsed/downloaded from the notebook's Output tab.")
if not HAS_LABELS:
    print("\n  This sequence has no labels, so only voxel_store/ and maps_2d/")
    print("  were populated -- that is expected, not a failure.")
print("\n  To process the full sequence, set TEST_MODE = False in the")
print("  'Frame discovery / test configuration' cell above and re-run.")


RUN SUMMARY
  Sequence                     : 00
  Mode                         : TEST
  Labels available (HAS_LABELS): True
  Frames discovered            : 4,541
  Frames attempted this run    : 10
  Frames voxelized             : 10
  Frames projected to 2.5D     : 10
  Frames label-propagated      : 10
  Frames failed (any stage)    : 0

  Output directories (/kaggle/working/):
    /kaggle/working/voxel_store                      10 files,    13.98 MB
    /kaggle/working/maps_2d                          10 files,     7.20 MB
    /kaggle/working/propagated_labels               120 files,    17.87 MB
    /kaggle/working/results                      

  Run summary saved to: /kaggle/working/results/run_summary.json

PERSISTENCE NOTE
  Everything written under /kaggle/working/ (voxel_store/, maps_2d/,
  propagated_labels/, results/) is automatically kept as this
  notebook's output once you run:
      Save Version -> Save & Run All (Commit)
  You do not need to manually export or downlo

In [50]:
# ============================================================
# 12. OPTIONAL: PACKAGE OUTPUT AS A REUSABLE KAGGLE DATASET
# ============================================================
# ------------------------------------------------------------------
# This cell is OPTIONAL and does NOT run automatically as part of the
# pipeline above. It is here so you can, after a full processing run,
# turn everything under /kaggle/working/ into a new reusable Kaggle
# Dataset -- without re-running this every time the notebook executes.
#
# Set RUN_DATASET_PACKAGING = True and run this cell manually when you
# are ready. It uses Kaggle's own supported dataset-upload mechanism
# (a dataset-metadata.json + the Kaggle API's "datasets create/version"
# commands), the same thing the Kaggle "New Dataset" UI does under the
# hood -- it does NOT invent a custom upload path.
#
# Two ways to do this on Kaggle, pick whichever is easier for you:
#   A) Manual (simplest, no credentials needed): after committing this
#      notebook (Save Version), open the notebook's "Output" tab and
#      use the "New Dataset" button there to publish /kaggle/working/
#      as a dataset directly from the UI.
#   B) Programmatic (this cell): requires your Kaggle API credentials
#      to be available to the kernel (e.g. added as Kaggle Secrets and
#      exported as KAGGLE_USERNAME / KAGGLE_KEY environment variables).
# ------------------------------------------------------------------
RUN_DATASET_PACKAGING = False   # <-- set True and run manually when ready
DATASET_TITLE         = "adaptive-voxelization-output"   # <-- edit
DATASET_SLUG          = "adaptive-voxelization-output"   # <-- edit, lowercase-hyphenated

if RUN_DATASET_PACKAGING:
    dataset_meta = {
        "title": DATASET_TITLE,
        "id": f"{os.environ.get('KAGGLE_USERNAME', '<your-kaggle-username>')}/{DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    meta_path = os.path.join(WORKING_ROOT, "dataset-metadata.json")
    with open(meta_path, "w") as f:
        json.dump(dataset_meta, f, indent=2)
    print(f"Wrote {meta_path}")

    have_creds = bool(os.environ.get("KAGGLE_USERNAME")) and bool(os.environ.get("KAGGLE_KEY"))
    if not have_creds:
        print("KAGGLE_USERNAME / KAGGLE_KEY are not set in the environment.")
        print("Add your Kaggle API credentials as Kaggle Secrets, or publish the")
        print("output manually from the notebook's Output tab instead (option A above).")
    else:
        # First time for this slug: 'datasets create'.
        # To push a new version of an existing dataset, use:
        #   kaggle datasets version -p /kaggle/working -m "update"
        os.system(f"kaggle datasets create -p {WORKING_ROOT} -r zip")
else:
    print("RUN_DATASET_PACKAGING is False -- skipping dataset upload.")
    print("Set RUN_DATASET_PACKAGING = True above and re-run this cell manually")
    print("once you are ready to publish /kaggle/working/ as a Kaggle Dataset.")


RUN_DATASET_PACKAGING is False -- skipping dataset upload.
Set RUN_DATASET_PACKAGING = True above and re-run this cell manually
once you are ready to publish /kaggle/working/ as a Kaggle Dataset.
